# 02. Unsupervised Anomaly Detection using Isolation Forest
**Power Grid AI - Predictive Maintenance**

### Objectives:
- Train and evaluate an `IsolationForest` model on multi-sensor transformer telemetry.
- Isolate sensor drift, unexpected thermal rises, cooling failures, and electrical discharge.
- Calculate continuous normalized anomaly scores.
- Benchmark detection accuracy against ground-truth validation events.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# Load cleaned and engineered data
data_path = os.path.join('..', 'data', 'processed', 'cleaned_transformer_data.csv')
df = pd.read_csv(data_path)

# Load trained model bundle
model_path = os.path.join('..', 'models', 'isolation_forest.pkl')
bundle = joblib.load(model_path)
model = bundle['model']
scaler = bundle['scaler']
features = bundle['features']

print(f"Loaded Isolation Forest model trained with features: {features}")


### Anomaly Score Inference

In [ ]:
X = scaler.transform(df[features])
raw_scores = model.decision_function(X)
min_s, max_s = raw_scores.min(), raw_scores.max()
anomaly_score = 1.0 - ((raw_scores - min_s) / (max_s - min_s + 1e-8))
preds = np.where(model.predict(X) == -1, 1, 0)

df['anomaly_score'] = anomaly_score
df['iso_pred'] = preds

print(f"Total points flagged as anomalies: {preds.sum()} out of {len(preds)} ({preds.mean()*100:.2f}%)")


### Model Performance & ROC Curve

In [ ]:
if 'anomaly_flag' in df.columns:
    print(classification_report(df['anomaly_flag'], df['iso_pred'], target_names=['Normal', 'Anomaly']))
    roc_auc = roc_auc_score(df['anomaly_flag'], df['anomaly_score'])
    
    fpr, tpr, _ = roc_curve(df['anomaly_flag'], df['anomaly_score'])
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f'Isolation Forest (AUC = {roc_auc:.4f})', color='darkorange', lw=2)
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Anomaly Detection ROC Curve', fontsize=12, fontweight='bold')
    plt.legend(loc='lower right')
    plt.show()


### Telemetry Anomaly Separation

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='load_percentage',
    y='winding_temperature_c',
    hue='iso_pred',
    palette={0: 'navy', 1: 'crimson'},
    alpha=0.7,
    s=40
)
plt.title('Transformer Operating Space: Normal vs Flagged Anomalies', fontsize=13, fontweight='bold')
plt.xlabel('Load Percentage (%)')
plt.ylabel('Winding Temperature (°C)')
plt.show()
